# Notebook 5 — CMF Budget-Shared Two-Stage Unlearning (Paper Protocol)

**Paper:** *An Illusion of Unlearning?* (Gao et al., AISTATS 2026 · arXiv:2604.08271v1)

**Protocol — matches paper exactly (Table 1 / Appendix A.3):**
- Forget set = **one entire class** (all ~5 000 training images of that class).
- Retain set = all training images from the remaining 9 classes.
- Loop over all 10 CIFAR-10 classes as the forget class; results averaged (mean ± std).
- **Output/Probe/NCC** evaluated on the **held-out test set** (paper Appendix A.2).

**Key difference from NB4:** SAME total epoch budget (per Table 4), SHARED between stages.
NB4's 4b ADDS epochs on top of a complete cmf_static run.
NB5 REALLOCATES: the last `k_shared` epochs of Stage 1 become Stage 2.

**Algorithm:**
- STAGE 1 — epochs 1 to (total - k_shared): cmf_static per-epoch loop.
- STAGE 2 — k_shared epochs: full model fine-tuning (encoder + W). No recompute_cmf during stage 2.

**Outputs per config:** `{method}_cmf_budgetshared_k{k}_{phase}_{src}_class{c}_seed{s}.pt`

In [ ]:
import subprocess, sys
def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout: print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-2000:])
    return r.returncode
sh('pip install -q timm einops scikit-learn matplotlib seaborn pytorch-lightning torchmetrics')

In [ ]:
import os, sys, json, random, math, time, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib; import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})
print('PyTorch:', torch.__version__, '  CUDA:', torch.cuda.is_available())

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'
if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} remote set-url origin https://github.com/tiensinh2/CMF_Unlearning.git')
    sh(f'git -C {REPO_DIR} pull origin main')
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
result = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'],
                        capture_output=True, text=True)
REPO_COMMIT = result.stdout.strip() or 'main'
print('Repo commit:', REPO_COMMIT)

In [ ]:
CKPT_DATASET_DIR = '/kaggle/input/datasets/btk23021592/cmf-notebook1'
_CONFIG_CANDIDATES = [
    f'{CKPT_DATASET_DIR}/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/checkpoints/cmf_benchmark/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/cmf_benchmark/cmf_benchmark_config.json',
]
config_path = CKPT_ROOT_NB1 = None
for _p in _CONFIG_CANDIDATES:
    if os.path.exists(_p):
        config_path = _p; CKPT_ROOT_NB1 = os.path.dirname(_p); break
assert config_path, f'Cannot find cmf_benchmark_config.json'
with open(config_path) as f: NB1_CFG = json.load(f)

DATASET     = NB1_CFG['dataset']       # 'cifar10'
ARCH        = NB1_CFG['arch']          # 'resnet18'
NUM_CLASSES = NB1_CFG['num_classes']   # 10
TEST_MODE   = NB1_CFG.get('test_mode', False)

# Paper protocol: sweep all 10 classes as forget class; one seed.
FORGET_CLASSES = list(range(NUM_CLASSES))   # [0, 1, ..., 9]
SEEDS          = [0]
THETA_O_SEED   = 0

# NB5 config
BASE_METHODS   = ['scrub', 'grad_ascent_descent', 'random_label', 'salun', 'tarun']
MEAN_SOURCES   = ['train']   # paper-faithful: class means from full training set
# Table 4: SCRUB+CMF = 3 epochs total. k_shared=1 leaves 2 Stage-1 epochs.
CMF_EPOCHS_BY_METHOD = {
    'random_label':        2 if TEST_MODE else 4,
    'salun':               2 if TEST_MODE else 4,
    'grad_ascent_descent': 1 if TEST_MODE else 3,
    'scrub':               1 if TEST_MODE else 3,
    'tarun':               1 if TEST_MODE else 3,
}
K_SHARED       = [1]   # extend to [1, 2] for sweep
PHASE2_DATA    = ['retain_plus_forget']   # phase 2 trains on retain + forget set
HPARAM_SOURCE  = 'table4'

# NB4 results dir (for combined table)
CKPT_ROOT_NB4 = '/kaggle/working/checkpoints/cmf_paper'
CKPT_ROOT     = '/kaggle/working/checkpoints/cmf_budgetshared_paper'
os.makedirs(f'{CKPT_ROOT}/stage1end', exist_ok=True)
os.makedirs(f'{CKPT_ROOT}/final',     exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'CMF_EPOCHS_BY_METHOD={CMF_EPOCHS_BY_METHOD}  K_SHARED={K_SHARED}  device={device}')

In [ ]:
import torchvision, torchvision.transforms as transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

full_train      = torchvision.datasets.CIFAR10('/kaggle/working/data', train=True,
                                               download=True,  transform=transform_train)
full_train_eval = torchvision.datasets.CIFAR10('/kaggle/working/data', train=True,
                                               download=False, transform=transform_test)
test_set = torchvision.datasets.CIFAR10('/kaggle/working/data', train=False,
                                        download=True, transform=transform_test)

# Pre-index by class label
test_targets = torch.tensor(test_set.targets)   # [10000]
TEST_CLASS_IDX = {
    c: (test_targets == c).nonzero(as_tuple=True)[0].tolist()
    for c in range(NUM_CLASSES)
}
train_targets = torch.tensor(full_train.targets)  # [50000]
TRAIN_CLASS_IDX = {
    c: (train_targets == c).nonzero(as_tuple=True)[0].tolist()
    for c in range(NUM_CLASSES)
}
print(f'Train: {len(full_train)}  Test: {len(test_set)}')

In [ ]:
import argparse
from unlearn.cmf_weights import ModelModule
from unlearn.cmf_two_stage import CMFWeightsTrainable
from unlearn import unlear_func

CMF_LR = {
    'scrub':               5e-3,
    'grad_ascent_descent': 1e-4,
    'random_label':        2e-3,
    'salun':               2e-3,
    'tarun':               5e-5,
}
CMF_BATCH = {'scrub': 64}
CMF_BATCH_DEFAULT = 128


def make_cmf_args(base_method, lr, epochs, mean_source, forget_class,
                  forget_train_idx, retain_train_idx, seed=0):
    return argparse.Namespace(
        dataset=DATASET, arch=ARCH, num_classes=NUM_CLASSES,
        class_label_names=list(range(NUM_CLASSES)),
        unlearn_method=f'{base_method}_CMF_RemoveFC',
        unlearn_class=[forget_class],
        batch_size=128, test_batch_size=256, lr=lr,
        momentum=0.9, weight_decay=5e-4, epochs_or_steps=epochs,
        seed=seed,
        num_retain_samples=len(retain_train_idx),
        num_forget_samples=len(forget_train_idx),
        grad_norm_clip=1.0,
        SVD_alpha_r=1000, SVD_alpha_f=30,
        SVD_samples=900, SVD_max_patches=10000,
        freeze_except_last=False,
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=epochs,
        salun_threshold=0.5,
        tarun_impair_lr=lr, tarun_samples_per_class=1000,
        dry_run=TEST_MODE, no_cuda=False, no_mps=True, gamma=0.5,
        data_path='/kaggle/working/data', remove_FC=True,
        CMFClassifier=True, CMF_momentum=0.9, pretrained=False, temperature=1.0,
        prob_batch_size=256, lp_every=0, mean_source=mean_source,
        repo_commit=REPO_COMMIT, test_mode=TEST_MODE,
    )


@torch.no_grad()
def eval_acc(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        correct += (model(x.to(device)).argmax(1).cpu() == y).sum().item()
        total   += y.size(0)
    return 100.0 * correct / max(total, 1)


@torch.no_grad()
def cmf_extract_features(model, loader):
    model.eval()
    feats, labs = [], []
    for x, y in loader:
        x = x.to(device)
        f = model.extract_features(x)
        z = model.extract_features(x)
        feats.append(z.cpu()); labs.append(y)
    return torch.cat(feats), torch.cat(labs)


def run_probe_cmf(model, train_retain_ldr, train_forget_ldr,
                  test_retain_ldr, test_forget_ldr, n_epochs=None):
    if n_epochs is None:
        n_epochs = 200 if DATASET.lower() == 'cifar100' else 50
    Xtr, ytr = cmf_extract_features(model, train_retain_ldr)
    Xfg, yfg = cmf_extract_features(model, train_forget_ldr)
    Xall = torch.cat([Xtr, Xfg])
    yall = torch.cat([ytr, yfg])
    head = nn.Linear(Xall.size(1), NUM_CLASSES).to(device)
    opt  = optim.SGD(head.parameters(), lr=1e-2, momentum=0.9)
    ldr  = torch.utils.data.DataLoader(
               torch.utils.data.TensorDataset(Xall, yall), batch_size=256, shuffle=True)
    for _ in range(n_epochs):
        head.train()
        for bx, by in ldr:
            opt.zero_grad()
            F.cross_entropy(head(bx.to(device)), by.to(device)).backward()
            opt.step()
    head.eval()
    with torch.no_grad():
        Xte_r, yte_r = cmf_extract_features(model, test_retain_ldr)
        Xte_f, yte_f = cmf_extract_features(model, test_forget_ldr)
        ret_acc = (head(Xte_r.to(device)).argmax(1).cpu() == yte_r).float().mean().item() * 100
        fgt_acc = (head(Xte_f.to(device)).argmax(1).cpu() == yte_f).float().mean().item() * 100
    return ret_acc, fgt_acc


def run_ncc_cmf(model, train_retain_ldr, train_forget_ldr,
                test_retain_ldr, test_forget_ldr):
    Xtr, ytr = cmf_extract_features(model, train_retain_ldr)
    Xfg, yfg = cmf_extract_features(model, train_forget_ldr)
    Xall = torch.cat([Xtr, Xfg])
    yall = torch.cat([ytr, yfg])
    means = []
    for c in range(NUM_CLASSES):
        mask = (yall == c)
        mu = Xall[mask].mean(0) if mask.any() else torch.zeros(Xall.size(1))
        means.append(mu)
    M = torch.stack(means)
    Xte_r, yte_r = cmf_extract_features(model, test_retain_ldr)
    Xte_f, yte_f = cmf_extract_features(model, test_forget_ldr)
    ret_pred = torch.cdist(Xte_r.unsqueeze(0), M.unsqueeze(0)).squeeze(0).argmin(1)
    fgt_pred = torch.cdist(Xte_f.unsqueeze(0), M.unsqueeze(0)).squeeze(0).argmin(1)
    return ((ret_pred == yte_r).float().mean().item() * 100,
            (fgt_pred == yte_f).float().mean().item() * 100)


def eval_cmf_three_metrics(model,
                           test_retain_ldr, test_forget_ldr,
                           train_retain_eval_ldr, train_forget_eval_ldr):
    out_ret = eval_acc(model, test_retain_ldr)
    out_fgt = eval_acc(model, test_forget_ldr)
    lp_ret, lp_fgt   = run_probe_cmf(model, train_retain_eval_ldr, train_forget_eval_ldr,
                                      test_retain_ldr, test_forget_ldr)
    ncc_ret, ncc_fgt = run_ncc_cmf(model, train_retain_eval_ldr, train_forget_eval_ldr,
                                    test_retain_ldr, test_forget_ldr)
    return {'output_retain_acc': out_ret, 'output_forget_acc': out_fgt,
            'probe_retain_acc':  lp_ret,  'probe_forget_acc':  lp_fgt,
            'ncc_retain_acc':    ncc_ret, 'ncc_forget_acc':    ncc_fgt}


print('Helpers ready.')

In [ ]:
# ─── NB5: Budget-shared two-stage loop ────────────────────────────────────────

# Load theta_o once
theta_o_path = f'{CKPT_ROOT_NB1}/pre_train/theta_o_seed{THETA_O_SEED}.pt'
if TEST_MODE: theta_o_path = theta_o_path.replace('.pt', '_testmode.pt')
assert os.path.exists(theta_o_path), f'Missing theta_o: {theta_o_path}'
ck_o = torch.load(theta_o_path, map_location=device)
theta_o_state = ck_o.get('model_state_dict', ck_o)

results_nb5 = []

for forget_class in FORGET_CLASSES:
    for seed in SEEDS:

        # ── Build whole-class splits ──────────────────────────────────────────
        forget_train_idx = TRAIN_CLASS_IDX[forget_class]
        retain_train_idx = [
            i for c in range(NUM_CLASSES)
            if c != forget_class
            for i in TRAIN_CLASS_IDX[c]
        ]

        retain_loader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train, retain_train_idx),
            batch_size=128, shuffle=True, num_workers=2)
        forget_loader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train, forget_train_idx),
            batch_size=128, shuffle=True, num_workers=2)
        full_train_loader = torch.utils.data.DataLoader(
            full_train, batch_size=256, shuffle=False, num_workers=2)

        train_retain_eval_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train_eval, retain_train_idx),
            batch_size=256, shuffle=False, num_workers=2)
        train_forget_eval_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train_eval, forget_train_idx),
            batch_size=256, shuffle=False, num_workers=2)

        test_forget_idx = TEST_CLASS_IDX[forget_class]
        test_retain_idx = [
            i for c in range(NUM_CLASSES)
            if c != forget_class
            for i in TEST_CLASS_IDX[c]
        ]
        test_forget_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(test_set, test_forget_idx),
            batch_size=256, shuffle=False, num_workers=2)
        test_retain_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(test_set, test_retain_idx),
            batch_size=256, shuffle=False, num_workers=2)

        retain_set_sub = torch.utils.data.Subset(full_train, retain_train_idx)
        forget_set_sub = torch.utils.data.Subset(full_train, forget_train_idx)

        # Internal full test loader (for unlearn method internal eval)
        _test_loader_full = torch.utils.data.DataLoader(
            test_set, batch_size=256, shuffle=False, num_workers=2)

        for base_method in BASE_METHODS:
            for mean_source in MEAN_SOURCES:
                for k_shared in K_SHARED:
                    for phase2_data in PHASE2_DATA:
                        total_epochs  = CMF_EPOCHS_BY_METHOD.get(base_method, 3)
                        stage1_epochs = total_epochs - k_shared
                        if stage1_epochs < 1:
                            print(f'  Skipping: k_shared={k_shared} >= total_epochs={total_epochs}')
                            continue
                        tag_base = (f'{base_method}_cmf_budgetshared_k{k_shared}_'
                                    f'{phase2_data}_{mean_source}_class{forget_class}_seed{seed}')
                        if TEST_MODE: tag_base += '_testmode'

                        ckpt_s1 = f'{CKPT_ROOT}/stage1end/{tag_base}_stage1end.pt'
                        ckpt_s2 = f'{CKPT_ROOT}/final/{tag_base}_final.pt'

                        if os.path.exists(ckpt_s2):
                            print(f'[{tag_base}] final exists — skipping.')
                            ck = torch.load(ckpt_s2, map_location=device)
                            results_nb5.append(ck['metrics'])
                            continue

                        print(f'\n[{tag_base}] total={total_epochs} stage1={stage1_epochs} k={k_shared}')
                        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

                        lr    = CMF_LR.get(base_method, 1e-3)
                        batch = CMF_BATCH.get(base_method, CMF_BATCH_DEFAULT)
                        args  = make_cmf_args(base_method, lr, total_epochs, mean_source,
                                              forget_class, forget_train_idx, retain_train_idx,
                                              seed=seed)
                        args.batch_size = batch
                        if base_method == 'scrub':
                            args.scrub_del_bsz  = 64
                            args.scrub_sgda_bsz = 64

                        model = ModelModule(args).to(device)
                        model.encoder.load_state_dict(theta_o_state, strict=False)

                        mean_loader = full_train_loader if mean_source == 'train' else retain_loader
                        model.eval()
                        model.recompute_cmf(mean_loader, device=device)

                        dispatch_key = f'{base_method}_CMF_RemoveFC'
                        fn = unlear_func[dispatch_key]

                        # ── STAGE 1: cmf_static for (total - k) epochs ───────
                        t0 = time.time()
                        print(f'  Stage 1: {stage1_epochs} epochs')
                        try:
                            model = fn(
                                args=args, model=model, device=device,
                                retain_loader=retain_loader, forget_loader=forget_loader,
                                train_loader=mean_loader, test_loader=_test_loader_full,
                                optimizer=None, epochs=stage1_epochs,
                                test_forget_loader=test_forget_ldr,
                                train_dataset=full_train,
                                val_index=retain_train_idx,
                            )
                        except Exception as e:
                            import traceback; traceback.print_exc()
                            print(f'  ERROR Stage 1: {e}'); continue

                        model.eval()
                        model.recompute_cmf(mean_loader, device=device)

                        # Save Stage 1 checkpoint
                        s1_metrics = eval_cmf_three_metrics(
                            model, test_retain_ldr, test_forget_ldr,
                            train_retain_eval_ldr, train_forget_eval_ldr
                        )
                        s1_metrics.update({
                            'stage': 'stage1end', 'method': base_method,
                            'mean_source': mean_source, 'k_shared': k_shared,
                            'phase2_data': phase2_data,
                            'forget_class': forget_class, 'seed': seed,
                            'total_epochs': total_epochs, 'stage1_epochs': stage1_epochs,
                            'protocol': 'whole_class_single',
                        })
                        torch.save({
                            'model_state_dict': model.state_dict(),
                            'config': {
                                'base_method': base_method, 'mean_source': mean_source,
                                'total_epochs': total_epochs, 'k_shared': k_shared,
                                'stage1_epochs': stage1_epochs, 'stage': 'stage1end',
                                'dataset': DATASET, 'arch': ARCH, 'num_classes': NUM_CLASSES,
                                'forget_class': forget_class, 'seed': seed,
                                'protocol': 'whole_class_single',
                                'repo_commit': REPO_COMMIT, 'test_mode': TEST_MODE,
                            },
                            'seed': seed, 'metrics': s1_metrics,
                        }, ckpt_s1)
                        print(f'  [Stage1End] out R={s1_metrics["output_retain_acc"]:.2f}% '
                              f'F={s1_metrics["output_forget_acc"]:.2f}%  Saved {ckpt_s1}')

                        # ── STAGE 2: full model fine-tuning (encoder + W together)
                        print(f'  Stage 2: {k_shared} epochs of full model fine-tuning')

                        for p in model.parameters():
                            p.requires_grad_(True)

                        if phase2_data == 'retain_plus_forget':
                            from torch.utils.data import ConcatDataset
                            s2_ds  = ConcatDataset([retain_set_sub, forget_set_sub])
                            s2_ldr = torch.utils.data.DataLoader(
                                s2_ds, batch_size=128, shuffle=True)
                        else:
                            s2_ldr = retain_loader

                        opt_s2 = optim.SGD(model.parameters(),
                                           lr=CMF_LR.get(base_method, 1e-3),
                                           momentum=0.9, weight_decay=5e-4,
                                           nesterov=True)
                        s2_log = []

                        for ep in range(1, k_shared + 1):
                            model.train()
                            ep_loss = n_batches = 0
                            for xb, yb in s2_ldr:
                                xb, yb = xb.to(device), yb.to(device)
                                opt_s2.zero_grad()
                                loss, _ = model.forward_a((xb, yb), stage='train')
                                loss.backward()
                                opt_s2.step()
                                ep_loss  += loss.item()
                                n_batches += 1
                                if TEST_MODE: break

                            model.eval()
                            m = eval_cmf_three_metrics(
                                model, test_retain_ldr, test_forget_ldr,
                                train_retain_eval_ldr, train_forget_eval_ldr
                            )
                            print(f'  [S2 ep{ep}] out R={m["output_retain_acc"]:.2f}% '
                                  f'F={m["output_forget_acc"]:.2f}%  '
                                  f'probe R={m["probe_retain_acc"]:.2f}% '
                                  f'F={m["probe_forget_acc"]:.2f}%  '
                                  f'ncc R={m["ncc_retain_acc"]:.2f}% '
                                  f'F={m["ncc_forget_acc"]:.2f}%')
                            s2_log.append({'epoch': stage1_epochs + ep, **m})

                        # Final metrics
                        final_metrics = eval_cmf_three_metrics(
                            model, test_retain_ldr, test_forget_ldr,
                            train_retain_eval_ldr, train_forget_eval_ldr
                        )
                        wall_min = (time.time() - t0) / 60
                        final_metrics.update({
                            'stage': 'final', 'method': base_method,
                            'mean_source': mean_source, 'k_shared': k_shared,
                            'phase2_data': phase2_data,
                            'forget_class': forget_class, 'seed': seed,
                            'total_epochs': total_epochs,
                            'stage1_epochs': stage1_epochs,
                            'wall_clock_minutes': wall_min,
                            'protocol': 'whole_class_single',
                        })

                        torch.save({
                            'model_state_dict': model.state_dict(),
                            'config': {
                                'base_method': base_method, 'mean_source': mean_source,
                                'total_epochs': total_epochs, 'k_shared': k_shared,
                                'stage1_epochs': stage1_epochs,
                                'phase2_data': phase2_data, 'stage': 'final',
                                'dataset': DATASET, 'arch': ARCH, 'num_classes': NUM_CLASSES,
                                'forget_class': forget_class, 'seed': seed,
                                'protocol': 'whole_class_single',
                                'repo_commit': REPO_COMMIT, 'test_mode': TEST_MODE,
                            },
                            'seed': seed, 'metrics': final_metrics,
                            'stage2_log': s2_log,
                        }, ckpt_s2)
                        print(f'  Saved {ckpt_s2}')
                        results_nb5.append(final_metrics)

df_nb5 = pd.DataFrame(results_nb5)
df_nb5.to_csv(f'{CKPT_ROOT}/results_nb5_budgetshared.csv', index=False)
print('\n=== NB5 budget-shared results saved ===')

if not df_nb5.empty:
    metric_cols = ['output_retain_acc','output_forget_acc',
                   'probe_retain_acc','probe_forget_acc','ncc_retain_acc','ncc_forget_acc']
    summary = (
        df_nb5.groupby(['method', 'mean_source', 'stage', 'k_shared', 'phase2_data'])[metric_cols]
        .agg(['mean', 'std'])
        .round(2)
    )
    summary.columns = ['_'.join(c) for c in summary.columns]
    summary.to_csv(f'{CKPT_ROOT}/results_nb5_budgetshared_summary.csv')
    print('\n=== Mean across all forget classes (NB5 final) ===')
    finals = df_nb5[df_nb5['stage'] == 'final']
    if not finals.empty:
        print(finals.groupby(['method', 'mean_source', 'k_shared', 'phase2_data'])[metric_cols]
              .mean().round(2).to_string())

In [ ]:
# ─── Combined 4-condition comparison table for base_method=scrub ─────────────
print('\n=== Combined 4-condition table: base_method=scrub, mean_source=train ===')

METRIC_COLS = ['output_retain_acc','output_forget_acc',
               'probe_retain_acc','probe_forget_acc',
               'ncc_retain_acc','ncc_forget_acc']

def load_csv_if_exists(path):
    try: return pd.read_csv(path)
    except: return pd.DataFrame()

df_4a = load_csv_if_exists(f'{CKPT_ROOT_NB4}/results_4a_cmf_static.csv')
df_4b = load_csv_if_exists(f'{CKPT_ROOT_NB4}/results_4b_cmf_posthoc.csv')

def fmt_df(df, label):
    if df.empty: return f'{label}: (no data)'
    parts = [label]
    for c in METRIC_COLS:
        if c in df.columns:
            v = df[c].dropna()
            parts.append(f'{c}={v.mean():.1f}±{v.std():.1f}' if len(v) else f'{c}=N/A')
    return '  '.join(parts)

# Condition 1: 4a cmf_static
c1 = df_4a[(df_4a['method'] == 'scrub') & (df_4a['mean_source'] == 'train')] \
    if 'method' in df_4a.columns else pd.DataFrame()
print(fmt_df(c1, '4a cmf_static'))

# Condition 2: 4b k=10 retain_only
c2 = df_4b
for col, val in [('method', 'scrub'), ('mean_source', 'train'),
                  ('k_posthoc', 10), ('phase2_data', 'retain_only')]:
    if col in c2.columns: c2 = c2[c2[col] == val]
print(fmt_df(c2, '4b cmf_static+k10W'))

# Condition 3: NB5 stage1end
if not df_nb5.empty:
    c3 = df_nb5[(df_nb5['method'] == 'scrub') & (df_nb5['mean_source'] == 'train') &
                (df_nb5['stage'] == 'stage1end')]
    print(fmt_df(c3, 'NB5 stage1end'))
    # Condition 4: NB5 final
    c4 = df_nb5[(df_nb5['method'] == 'scrub') & (df_nb5['mean_source'] == 'train') &
                (df_nb5['stage'] == 'final')]
    print(fmt_df(c4, 'NB5 final'))

print('\nAll conditions use same θ_o, same splits, same evaluation protocol.')